# Import and Verifications

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sqlalchemy

In [2]:
print(sqlalchemy.__version__)

2.0.48


In [3]:
from sqlalchemy import create_engine
from sqlalchemy import text

# Create SQL Engine and Test Connection

In [4]:
engine = create_engine('postgresql+psycopg2://postgres@localhost/train_reward_compare')

In [5]:
with engine.connect() as conn:
    result = conn.execute(text("select count(*) from onet.occupation_data"))
    print(result.fetchone())

(1016,)


# Import BLS Files

In [6]:
file_nat = '../raw-data/bls-wage/oesm24nat/national_M2024_dl.xlsx'
file_sta = '../raw-data/bls-wage/oesm24st/state_M2024_dl.xlsx'
file_met = '../raw-data/bls-wage/oesm24ma/MSA_M2024_dl.xlsx'
file_bos = '../raw-data/bls-wage/oesm24ma/BOS_M2024_dl.xlsx'
file_nbs = '../raw-data/bls-wage/oesm24in4/natsector_M2024_dl.xlsx'

In [7]:
nat = pd.ExcelFile(file_nat)
sta = pd.ExcelFile(file_sta)
met = pd.ExcelFile(file_met)
bos = pd.ExcelFile(file_bos)
nbs = pd.ExcelFile(file_nbs)

# Test Data in Each Imported File

In [8]:
print(nat.sheet_names)
print(sta.sheet_names)
print(met.sheet_names)
print(bos.sheet_names)
print(nbs.sheet_names)

['national_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['state_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['MSA_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['BOS_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['Natsector_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']


# Rough Wrangle and Test National BLS Data

In [9]:
nat.parse('national_M2024_dl').head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN


In [10]:
nat_data = nat.parse('national_M2024_dl')

In [11]:
nat_data.columns = nat_data.columns.str.lower()

In [12]:
nat_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN


# Rough Wrangle National Field Descriptions

In [13]:
nat_desc = nat.parse('Field Descriptions')

In [14]:
nat_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [15]:
nat_desc = nat_desc.drop('Unnamed: 2', axis=1)

In [16]:
drop_index = nat_desc.index[0:8]

In [17]:
nat_desc_dropped = nat_desc.drop(drop_index)

In [18]:
nat_desc_dropped.head()

,May 2024 OEWS Estimates,Unnamed: 1
8,Field,Field Description
9,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
10,area_title,Area name
11,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
12,prim_state,"The primary state for the given area. ""US"" is ..."


In [19]:
nat_desc_new_header = nat_desc_dropped.iloc[0]

In [20]:
nat_desc_new = nat_desc_dropped[1:]

In [21]:
nat_desc_new.columns = nat_desc_new_header

In [22]:
nat_desc_new = nat_desc_new.reset_index(drop=True)

In [23]:
nat_desc_new.index.name = None

In [24]:
nat_desc_new = nat_desc_new.rename_axis(None, axis=1)

In [25]:
nat_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


# Rough Wrangle State Data

In [26]:
sta.parse('state_M2024_dl').head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.07,30.82,47.51,23520,30660,43830,64110,98810,NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,48.39,68.5,98.03,51100,72870,100640,142480,203900,NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.04,106.69,#,104950,130950,164400,221910,#,NaN,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,51.12,78.26,#,50410,74720,106330,162780,#,NaN,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18270,20950,26990,41760,63900,True,NaN


In [27]:
sta_data = sta.parse('state_M2024_dl')

In [28]:
sta_data.columns = sta_data.columns.str.lower()

In [29]:
sta_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.07,30.82,47.51,23520,30660,43830,64110,98810,NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,48.39,68.5,98.03,51100,72870,100640,142480,203900,NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.04,106.69,#,104950,130950,164400,221910,#,NaN,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,51.12,78.26,#,50410,74720,106330,162780,#,NaN,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18270,20950,26990,41760,63900,True,NaN


# Rough Wrangle State Field Descriptions

In [30]:
sta_desc = sta.parse('Field Descriptions')

In [31]:
sta_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [32]:
sta_desc = sta_desc.drop('Unnamed: 2', axis=1)
drop_index = sta_desc.index[0:8]
sta_desc_dropped = sta_desc.drop(drop_index)
sta_desc_new_header = sta_desc_dropped.iloc[0]
sta_desc_new = sta_desc_dropped[1:]
sta_desc_new.columns = sta_desc_new_header
sta_desc_new = sta_desc_new.reset_index(drop=True)
sta_desc_new.index.name = None
sta_desc_new = sta_desc_new.rename_axis(None, axis=1)

In [33]:
sta_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


# Rough Wrangle Metropolitan Data

In [34]:
met_data = met.parse('MSA_M2024_dl')
met_data.columns = met_data.columns.str.lower()

In [35]:
met_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,20.25,29.34,45.3,23120,29640,42120,61020,94220,NaN,NaN
1,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,42.03,61.17,84.3,44710,61250,87420,127230,175350,NaN,NaN
2,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,85.35,#,#,99140,107040,177520,#,#,NaN,NaN
3,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,37.18,58.64,87.22,35980,50640,77340,121970,181420,NaN,NaN
4,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,...,50.02,73.86,99.9,58480,82560,104030,153630,207790,NaN,NaN


# Rough Wrangle Metropolitan Field Descriptions

In [36]:
met_desc = met.parse('Field Descriptions')

In [37]:
met_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [38]:
met_desc = met_desc.drop('Unnamed: 2', axis=1)
drop_index = met_desc.index[0:8]
met_desc_dropped = met_desc.drop(drop_index)
met_desc_new_header = met_desc_dropped.iloc[0]
met_desc_new = met_desc_dropped[1:]
met_desc_new.columns = met_desc_new_header
met_desc_new = met_desc_new.reset_index(drop=True)
met_desc_new.index.name = None
met_desc_new = met_desc_new.rename_axis(None, axis=1)

In [39]:
met_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


# Rough Wrangle Balance of State Data

In [40]:
bos_data = bos.parse('BOS_M2024_dl')
bos_data.columns = bos_data.columns.str.lower() 

In [41]:
bos_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,18.94,25,35.51,22380,29690,39400,52000,73870,NaN,NaN
1,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,40.54,58.5,80.72,44770,60730,84320,121680,167910,NaN,NaN
2,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,71.58,85.16,#,114610,121930,148890,177130,#,NaN,NaN
3,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,46.63,73.99,101.78,42680,63380,96980,153900,211700,NaN,NaN
4,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18210,22690,51460,52020,52020,True,NaN


# Rough Wrangle Balance of State Field Descriptions

In [42]:
bos_desc = bos.parse('Field Descriptions')

In [43]:
bos_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [44]:
bos_desc = bos_desc.drop('Unnamed: 2', axis=1)
drop_index = bos_desc.index[0:8]
bos_desc_dropped = bos_desc.drop(drop_index)
bos_desc_new_header = bos_desc_dropped.iloc[0]
bos_desc_new = bos_desc_dropped[1:]
bos_desc_new.columns = bos_desc_new_header
bos_desc_new = bos_desc_new.reset_index(drop=True)
bos_desc_new.index.name = None
bos_desc_new = bos_desc_new.rename_axis(None, axis=1)

In [45]:
bos_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


# Rough Wrangle National Sector Data

In [46]:
nbs_data = nbs.parse('Natsector_M2024_dl')
nbs_data.columns = nbs_data.columns.str.lower() 

In [47]:
nbs_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,00-0000,All Occupations,...,17.66,22.21,30.23,33280,34680,36720,46200,62880,NaN,NaN
1,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-0000,Management Occupations,...,46.08,63.4,90.74,41710,67090,95840,131870,188730,NaN,NaN
2,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-1000,Top Executives,...,40.91,61.6,91.98,32100,57390,85080,128140,191320,NaN,NaN
3,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-1010,Chief Executives,...,103.99,#,#,105020,154280,216290,#,#,NaN,NaN
4,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-1011,Chief Executives,...,103.99,#,#,105020,154280,216290,#,#,NaN,NaN


# Rough Wrangle National Sector Field Descriptions

In [48]:
nbs_desc = nbs.parse('Field Descriptions')

In [49]:
nbs_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [50]:
nbs_desc = nbs_desc.drop('Unnamed: 2', axis=1)
drop_index = nbs_desc.index[0:8]
nbs_desc_dropped = nbs_desc.drop(drop_index)
nbs_desc_new_header = nbs_desc_dropped.iloc[0]
nbs_desc_new = nbs_desc_dropped[1:]
nbs_desc_new.columns = nbs_desc_new_header
nbs_desc_new = nbs_desc_new.reset_index(drop=True)
nbs_desc_new.index.name = None
nbs_desc_new = nbs_desc_new.rename_axis(None, axis=1)

In [51]:
nbs_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...
